In [2]:
import torch

words = open('names.txt', 'r').read().splitlines()
# 构建词表，先统计name里面的字母，sort set去重
chars = sorted(list(set(''.join(words))))
# 2. 建立字符到整数的映射 (s_to_i) 编码
s_to_i = {s:i+1 for i, s in enumerate(chars)}
s_to_i['.'] = 0
# 3. 建立整数到字符的反向映射 (i_to_s) 解码
i_to_s = {i: s for s, i in s_to_i.items()}

xs, ys = [], []
for w in words[:1]:
    chs = ['.'] + list(w) + ['.']
    # 输入ch1 预测下一个ch2
    for ch1, ch2 in zip(chs, chs[1:]):
        # 找到对应编号
        ix1 = s_to_i[ch1]
        ix2 = s_to_i[ch2]
        print(ch1, ch2)
        xs.append(ix1)
        ys.append(ix2)
xs = torch.tensor(xs)
ys = torch.tensor(ys)

. e
e m
m m
m a
a .


In [3]:
xs

tensor([ 0,  5, 13, 13,  1])

In [4]:
ys

tensor([ 5, 13, 13,  1,  0])

In [5]:
# 加载独热编码
import torch.nn.functional as F

xenc = F.one_hot(xs, num_classes=27).float()
xenc

tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [8]:
# 前向传播部分：
# 定义w的初始权重，因为输入维度是27维的独热向量，输出是27个字母的预测
w = torch.randn((27,27))
# 未归一化的原始打分，有正有负，数值越大，代表模型认为下一个 Token 出现的置信度越高
# 需要经过softmax算子转换成概率
logits = xenc @ w
# (5, 27) @ (27, 27)
counts = logits.exp()
probs = counts / counts.sum(1, keepdim=True)
loss = -probs[torch.arange(5), ys].log().mean()

In [9]:
# 反向传播部分
w.grad = None
loss.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [10]:
w.data += -0.1 * w.grad

TypeError: unsupported operand type(s) for *: 'float' and 'NoneType'

In [11]:
# 整体写下来：
xs, ys = [], []
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = s_to_i[ch1]
        ix2 = s_to_i[ch2]
        xs.append(ix1)
        ys.append(ix2)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of example:', num)

# 初始化网络
g = torch.Generator().manual_seed(2147483647)
w = torch.randn((27,27), generator=g, requires_grad=True)

number of example: 228146


In [ ]:
# gradient
for k in range(10):
    # 输入的独热向量
    xenc = F.one_hot(xs, num_classes=27).float()
    # 输出的未归一化的logits
    logits = xenc @ w
    # 指数
    counts = logits.exp()
    # 计算概率
    probs = counts/ counts.sum(1, keepdim=True)
    # 计算loss
    loss = - probs[torch.arange(num), ys].log().mean()
    print(loss.item())
    w.grad = None
    # 反向传播
    loss.backward()
    # 更新参数
    w.data += -50 * w.grad

3.758953809738159
3.371098518371582
3.1540417671203613
3.020373582839966
2.9277119636535645
2.860402822494507
2.80972957611084
2.7701025009155273
2.7380733489990234
2.711496591567993
